# Dedup (item) — news pipeline eval notebook

Tests the `/dedup/item` endpoint (single-item call) against a **synthetic Weaviate collection**
created and deleted by this notebook. Corpus items are embedded via `/vectorizer/embed` before
being upserted, so cosine similarities are semantically meaningful.

**Decision logic (two-step)**:

| Band | Condition | Action |
|------|-----------|--------|
| High | score ≥ `MATCH_THRESHOLD` (0.92) | auto-match, no LLM |
| Mid  | `LLM_LOW` (0.75) ≤ score < `MATCH_THRESHOLD` | LLM adjudication — `unsure` → no_match (conservative) |
| Low  | score < `LLM_LOW` | auto-no_match, no LLM |
| —    | 0 candidates | no_match |

**Edge cases covered**: empty corpus, auto-match (exact same text), auto-no_match (unrelated
topic), LLM band (genuine paraphrase), LLM band (near-miss), `type_filter` isolation.

> Advisory cases (LLM band) require Ollama to be running.  
> `WEAVIATE_URL` must resolve to the same Weaviate instance the NLP service uses internally.

In [ ]:
import os, sys, logging
import requests

# --- endpoints -----------------------------------------------------------
NLP_BASE_URL   = os.environ.get('NLP_BASE_URL', 'https://wiig.dia.fi.upm.es/nlp').rstrip('/')
WEAVIATE_URL   = os.environ.get('WEAVIATE_URL', 'http://localhost:8080').rstrip('/')
VECTORIZER_URL = os.environ.get('VECTORIZER_URL', f'{NLP_BASE_URL}/vectorizer/embed')
ENDPOINT       = f'{NLP_BASE_URL}/dedup/item'
HEADERS        = {'Content-Type': 'application/json',
                  'X-API-Key': os.environ.get('NLP_API_KEY', '')}
TIMEOUT        = 60

# --- service thresholds (mirror nlp/dedup/service.py defaults) ----------
MATCH_THRESHOLD = float(os.environ.get('DEDUP_ITEM_MATCH',   '0.92'))
LLM_LOW         = float(os.environ.get('DEDUP_ITEM_LLM_LOW', '0.75'))
TOP_K           = int(os.environ.get('DEDUP_TOP_K', '10'))

COLLECTION       = 'DedupItemEval'
EMPTY_COLLECTION = 'DedupItemEvalEmpty'

# --- logging -------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(name)-24s %(levelname)-8s  %(message)s',
    stream=sys.stdout, force=True,
)
for _noisy in ('httpx', 'httpcore', 'urllib3', 'requests', 'charset_normalizer'):
    logging.getLogger(_noisy).setLevel(logging.WARNING)
log = logging.getLogger('eval.dedup.item')

print(f'NLP        → {ENDPOINT}')
print(f'VECTORIZER → {VECTORIZER_URL}')
print(f'WAV        → {WEAVIATE_URL}  (must be the same instance the NLP service sees)')
print(f'thresholds:  match ≥ {MATCH_THRESHOLD}   llm_low ≥ {LLM_LOW}   top_k = {TOP_K}')

In [ ]:
# ── SETUP: create Weaviate collections and insert test corpus ─────────────

# --- Weaviate v1 REST helpers -------------------------------------------
def _wv(path, method='GET', body=None):
    url = f'{WEAVIATE_URL}/v1/{path}'
    fn  = {'GET': requests.get, 'POST': requests.post, 'DELETE': requests.delete}[method]
    kw  = {'timeout': 15}
    if body is not None:
        kw['json'] = body
    r = fn(url, **kw)
    r.raise_for_status()
    return r.json() if r.content else {}


def _ensure_collection(name, drop_existing=False):
    schema = {
        'class': name, 'vectorizer': 'none',
        'properties': [
            {'name': 'summary', 'dataType': ['text']},
            {'name': 'type',    'dataType': ['text']},
        ],
    }
    try:
        _wv('schema', 'POST', schema)
        log.info('created collection  %s', name)
    except requests.HTTPError as e:
        if e.response.status_code == 422 and drop_existing:
            log.warning('%s already exists — dropping and re-creating', name)
            _wv(f'schema/{name}', 'DELETE')
            _wv('schema', 'POST', schema)
        else:
            raise


def _insert(collection, props, vector):
    r = _wv('objects', 'POST',
            {'class': collection, 'properties': props, 'vector': vector})
    return r['id']


# --- Vectorizer ----------------------------------------------------------
def embed(text):
    """Call /vectorizer/embed and return the embedding as a plain list."""
    r = requests.post(VECTORIZER_URL, json={'text': text}, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()['embedding']


# --- Corpus: 3 articles + 1 entity --------------------------------------
_CORPUS = [
    ('c1', 'article',
     'El Ayuntamiento de Madrid aprueba la ampliación del carril bici en la Gran Vía '
     'con un presupuesto de 2,4 millones de euros. Las obras comenzarán en junio.'),
    ('c2', 'article',
     'Barcelona lidera el ranking nacional de movilidad sostenible. '
     'La capital catalana supera a Madrid en kilómetros de carril bici '
     'según un estudio reciente.'),
    ('c3', 'article',
     'El Ministerio de Transportes presenta la Estrategia Nacional de '
     'Movilidad Ciclista 2026-2030, dotada con 2.000 millones de euros '
     'para municipios de toda España.'),
    ('c4', 'entity',
     'Organismo de gobierno local responsable de la gestión municipal '
     'y los servicios urbanos de la ciudad de Madrid.'),
]

# Connectivity sanity check
try:
    _wv('.well-known/ready')
    log.info('Weaviate reachable')
except Exception as exc:
    log.error('Weaviate not reachable at %s  →  %s', WEAVIATE_URL, exc)
    raise

# Create collections
_ensure_collection(COLLECTION,       drop_existing=True)
_ensure_collection(EMPTY_COLLECTION, drop_existing=True)   # stays empty (edge case)

# Embed and insert corpus
IDS   = {}   # key → weaviate object id
TEXTS = {}   # key → summary text (used for query embeddings in cases)

for _key, _type, _summary in _CORPUS:
    vec = embed(_summary)
    oid = _insert(COLLECTION, {'summary': _summary, 'type': _type}, vec)
    IDS[_key]   = oid
    TEXTS[_key] = _summary
    log.info('inserted  %s  type=%-8s  id=%s…  dim=%d', _key, _type, oid[:8], len(vec))

print(f'\n{COLLECTION:<24} {len(_CORPUS)} objects  (3 articles, 1 entity)')
print(f'{EMPTY_COLLECTION:<24} 0 objects  (empty-corpus edge case)')

In [ ]:
# --- Service call + trace render + scoring helpers ----------------------

def call_dedup_item(embedding, *, collection=COLLECTION, compare_text='',
                    kind='article', compare_property='summary', type_filter=None):
    body = {
        'collection':       collection,
        'embedding':        embedding,
        'kind':             kind,
        'compare_text':     compare_text,
        'compare_property': compare_property,
    }
    if type_filter is not None:
        body['type_filter'] = type_filter
    r = requests.post(ENDPOINT, json=body, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def _classify_path(score, n_cands):
    if n_cands == 0:             return 'empty-corpus'
    if score >= MATCH_THRESHOLD: return 'auto-match'
    if score < LLM_LOW:          return 'auto-no_match'
    return 'llm-band'


def _path_label(score, n_cands):
    p = _classify_path(score, n_cands)
    if p == 'empty-corpus':   return 'empty-corpus → no_match (0 candidates)'
    if p == 'auto-match':     return f'auto-match        score={score:.4f} ≥ {MATCH_THRESHOLD}'
    if p == 'auto-no_match':  return f'auto-no_match     score={score:.4f} < {LLM_LOW}'
    return                           f'LLM adjudication  score={score:.4f} ∈ [{LLM_LOW}, {MATCH_THRESHOLD})'


def render(resp):
    decision  = resp['decision']
    target_id = resp.get('target_id')
    score     = resp.get('score', 0.0)
    cands     = resp.get('candidates', [])

    path = _path_label(score, len(cands))
    log.info('PATH    %s', path)
    log.info('RESULT  decision=%-10s  target=%s',
             decision, target_id[:8] + '…' if target_id else 'None')

    print(f'  candidates : {len(cands)}')
    for i, c in enumerate(cands):
        corp_key = next((k for k, oid in IDS.items() if oid == c['id']), '?')
        print(f'    [{i}] {corp_key}  score={c["score"]:.4f}  id={c["id"][:8]}…')
    print(f'  decision   : {decision}'
          + (f'  → {target_id[:8]}…' if target_id else ''))
    print(f'  path       : {path}')


# --- Cases ---------------------------------------------------------------
# query embeddings are produced by the vectorizer — same model the service uses
CASES = [
    {
        'id':          'dedup-001',
        'description': 'empty corpus — 0 candidates, immediate no_match',
        'embedding':   embed('Texto de prueba en colección vacía.'),
        'params':      {'collection': EMPTY_COLLECTION,
                        'compare_text': 'Texto de prueba en colección vacía.'},
        'expected':    'no_match',
        'notes':       'WeaviateIndex.near returns [] → early-exit in decide_item',
    },
    {
        'id':          'dedup-002',
        'description': 'auto-match — query is the exact stored text of c1 (sim ≈ 1.0)',
        'embedding':   embed(TEXTS['c1']),
        'params':      {'compare_text': TEXTS['c1']},
        'expected':    'match',
    },
    {
        'id':          'dedup-003',
        'description': 'auto-no_match — unrelated topic, expected sim << 0.75',
        'embedding':   embed('Récord histórico de ventas de vehículos eléctricos en Europa '
                             'durante el primer trimestre, liderado por el mercado alemán.'),
        'params':      {'compare_text': 'Récord histórico de ventas de vehículos eléctricos '
                                        'en Europa durante el primer trimestre.'},
        'expected':    'no_match',
    },
    {
        'id':          'dedup-004',
        'description': 'LLM band — genuine duplicate, paraphrase of c1  [advisory]',
        'embedding':   embed('El Consistorio madrileño da luz verde a la ampliación del '
                             'carril bici en la Gran Vía. La inversión asciende a 2,4 '
                             'millones de euros y las obras arrancan en junio.'),
        'params':      {'compare_text': (
            'El Consistorio madrileño da luz verde a la ampliación del carril bici '
            'en la Gran Vía. La inversión asciende a 2,4 millones de euros y las obras '
            'arrancan en junio.'
        )},
        'expected':    'match',
        'advisory':    True,
    },
    {
        'id':          'dedup-005',
        'description': 'LLM band — near-miss, same context, different event  [advisory]',
        'embedding':   embed('El Ayuntamiento de Madrid aprueba la ampliación de aceras '
                             'peatonales en el centro histórico con una inversión de '
                             '1,8 millones de euros para el verano.'),
        'params':      {'compare_text': (
            'El Ayuntamiento de Madrid aprueba la ampliación de aceras peatonales '
            'en el centro histórico con una inversión de 1,8 millones de euros '
            'para el verano.'
        )},
        'expected':    'no_match',
        'advisory':    True,
    },
    {
        'id':          'dedup-006',
        'description': 'type_filter=entity — article candidates excluded; c4 (entity) matches',
        'embedding':   embed(TEXTS['c4']),
        'params':      {'compare_text': TEXTS['c4'],
                        'kind':         'entity',
                        'compare_property': 'summary',
                        'type_filter':  'entity'},
        'expected':    'match',
    },
    {
        'id':          'dedup-007',
        'description': 'type_filter=article — entity c4 excluded; article sims expected < 0.75',
        'embedding':   embed(TEXTS['c4']),
        'params':      {'compare_text': TEXTS['c4'],
                        'kind':         'article',
                        'compare_property': 'summary',
                        'type_filter':  'article'},
        'expected':    'no_match',
        'notes':       'c4 is filtered out by type; article corpus is about urban cycling '
                       '(different semantic space from municipal governance) → low sim',
        'advisory':    True,
    },
]

print(f'{len(CASES)} cases')
for c in CASES:
    adv = '  [advisory]' if c.get('advisory') else ''
    print(f"  {c['id']}: {c['description']}{adv}")

In [ ]:
results = []   # (case_id, passed, advisory, resp_or_None)

for case in CASES:
    adv_tag = ' [advisory]' if case.get('advisory') else ''
    print(f"── {case['id']}: {case['description']}{adv_tag}")
    try:
        resp = call_dedup_item(case['embedding'], **case['params'])
    except Exception as exc:
        print(f'   ERROR: {exc}')
        if case.get('notes'):
            print(f'   note: {case["notes"]}')
        print()
        results.append((case['id'], False, case.get('advisory', False), None))
        continue

    render(resp)

    passed = resp['decision'] == case['expected']
    icon   = '✅' if passed else ('⚠️ ' if case.get('advisory') else '❌')
    label  = 'pass' if passed else ('advisory mismatch' if case.get('advisory') else 'FAIL')
    print(f'   {icon} {label}  expected={case["expected"]}  got={resp["decision"]}')
    if case.get('notes'):
        print(f'   note: {case["notes"]}')
    print()
    results.append((case['id'], passed, case.get('advisory', False), resp))

In [ ]:
# ── SCORECARD + METRICS ───────────────────────────────────────────────────

case_map = {c['id']: c for c in CASES}

def _path_tag(resp):
    if resp is None: return 'error'
    return _classify_path(resp.get('score', 0.0), len(resp.get('candidates', [])))

hard = [(cid, ok, resp) for cid, ok, adv, resp in results if not adv]
adv  = [(cid, ok, resp) for cid, ok, adv, resp in results if adv]
total_n = len(results)

# Confusion matrix (hard cases)
tp = sum(1 for cid, ok, resp in hard
         if resp and resp['decision'] == 'match'    and case_map[cid]['expected'] == 'match')
fp = sum(1 for cid, ok, resp in hard
         if resp and resp['decision'] == 'match'    and case_map[cid]['expected'] == 'no_match')
fn = sum(1 for cid, ok, resp in hard
         if resp and resp['decision'] == 'no_match' and case_map[cid]['expected'] == 'match')
tn = sum(1 for cid, ok, resp in hard
         if resp and resp['decision'] == 'no_match' and case_map[cid]['expected'] == 'no_match')

# Path distribution
path_counts = {}
for _, _, _, resp in results:
    p = _path_tag(resp)
    path_counts[p] = path_counts.get(p, 0) + 1
llm_n = path_counts.get('llm-band', 0)

hard_passed = sum(1 for _, ok, _ in hard if ok)
adv_passed  = sum(1 for _, ok, _ in adv  if ok)

print('SCORECARD · /dedup/item')
print(f'  Hard cases passing   : {hard_passed}/{len(hard)}')
print(f'  Advisory (LLM) cases : {adv_passed}/{len(adv)}')
print(f'    (advisory mismatches are informational — LLM verdict is the ground truth)')
print()
print('DECISION PATH DISTRIBUTION')
for path, count in sorted(path_counts.items()):
    pct = 100.0 * count / total_n if total_n else 0
    print(f'  {path:<20} {count:2d} / {total_n}  ({pct:.0f}%)')
if total_n:
    print(f'  LLM invocation rate  {llm_n:2d} / {total_n}  ({100.0*llm_n/total_n:.0f}%)')
print()
print('CONFUSION MATRIX  (hard cases only)')
print(f'                  predicted match  predicted no_match')
print(f'  actual match         {tp:2d}                {fn:2d}')
print(f'  actual no_match      {fp:2d}                {tn:2d}')
if tp + fp > 0: print(f'  precision  {tp/(tp+fp):.2f}')
if tp + fn > 0: print(f'  recall     {tp/(tp+fn):.2f}')
print()
print('PER-CASE')
for cid, ok, adv_flag, resp in results:
    icon    = '✅' if ok else ('⚠️ ' if adv_flag else '❌')
    path    = _path_tag(resp)
    score_s = f'  score={resp["score"]:.4f}' if resp else ''
    print(f'  {icon} {cid:<12}  [{path}]{score_s}')

In [ ]:
# ── TEARDOWN: delete test collections ─────────────────────────────────────
for _col in (COLLECTION, EMPTY_COLLECTION):
    try:
        _wv(f'schema/{_col}', 'DELETE')
        log.info('deleted collection  %s', _col)
    except requests.HTTPError as exc:
        log.warning('could not delete %s: %s', _col, exc)
print('teardown complete')